# OpenOppsDB — Snapshot health

Coverage, freshness, sync runs, and observation mix for the attached snapshot. Chart-first.

## Contents

- Setup (read-only `/kaggle/input`, `mode=ro&immutable=1`)
- Queries and charts for this kernel
- Links to the rest of the collection

## Collection

| Notebook | Kernel | What it is for |
| --- | --- | --- |
| **Starter** | [`wyattowalsh/openoppsdb-starter-notebook`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-starter-notebook) | Front door: tables, recent open jobs, first `%%sql` cells |
| **Explorer (featured)** | [`wyattowalsh/openoppsdb-explorer`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-explorer) | Gradio UI: jobs, companies, skills, filters/plots |
| **Advanced usage** | [`wyattowalsh/openoppsdb-advanced-usage`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-advanced-usage) | Joins, version history, company drill-down, Parquet |
| **SQL playground** | [`wyattowalsh/openoppsdb-sql-playground`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-sql-playground) | JupySQL studio: CTEs, DuckDB attach, Parquet scans |
| **Hiring market map** | [`wyattowalsh/openoppsdb-hiring-market-map`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-hiring-market-map) | Company, provider, location, and remote mix charts |
| **Skills radar** | [`wyattowalsh/openoppsdb-skills-radar`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-skills-radar) | Skill groups, keywords, and co-occurrence |
| **Snapshot health** | [`wyattowalsh/openoppsdb-snapshot-health`](https://www.kaggle.com/code/wyattowalsh/openoppsdb-snapshot-health) | Coverage, freshness, sync runs, observation mix |


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"
ROUTE_LEDGER = {
    "pine": "#2f6f50",
    "paper": "#f7f1df",
    "brass": "#d99629",
    "ink": "#1d281f",
    "info": "#336d8f",
}

db_candidates = sorted(Path("/kaggle/input").glob("**/openoppsdb.sqlite"))
if not db_candidates:
    raise FileNotFoundError("No openoppsdb.sqlite input found under /kaggle/input")
DB_PATH = db_candidates[0]
DATASET_DIR = DB_PATH.parent
DB_URI = f"file:{DB_PATH}?mode=ro&immutable=1"
PARQUET_DIR = DATASET_DIR / "exports" / "parquet"
print(f"Reading OpenOppsDB snapshot from {DB_PATH}")


In [ ]:
with sqlite3.connect(DB_URI, uri=True) as conn:
    run_mix = pd.read_sql_query(
        """
        select status, count(*) as runs
        from job_sync_runs
        group by status
        order by runs desc
        """,
        conn,
    )
    observation_mix = pd.read_sql_query(
        """
        select observation_kind, count(*) as observations
        from job_sync_observations
        group by observation_kind
        order by observations desc
        """,
        conn,
    )
    freshness = pd.read_sql_query(
        """
        select status, min(last_seen_at) as oldest_last_seen,
               max(last_seen_at) as newest_last_seen, count(*) as jobs
        from jobs
        group by status
        order by jobs desc
        """,
        conn,
    )
    coverage = pd.read_sql_query(
        """
        select count(distinct source_id) as sources,
               count(distinct board_key) as boards,
               count(*) as open_jobs
        from jobs
        where status = 'open'
        """,
        conn,
    )

display(coverage)
display(freshness)
display(run_mix)
observation_mix


In [ ]:
if not observation_mix.empty:
    fig = px.bar(
        observation_mix,
        x="observation_kind",
        y="observations",
        color_discrete_sequence=[ROUTE_LEDGER["pine"]],
        title="Observation mix",
    )
    fig.update_layout(
        paper_bgcolor=ROUTE_LEDGER["paper"], font_color=ROUTE_LEDGER["ink"]
    )
    fig.show()
if not run_mix.empty:
    fig2 = px.bar(
        run_mix,
        x="status",
        y="runs",
        color_discrete_sequence=[ROUTE_LEDGER["brass"]],
        title="Sync run status",
    )
    fig2.update_layout(
        paper_bgcolor=ROUTE_LEDGER["paper"], font_color=ROUTE_LEDGER["ink"]
    )
    fig2.show()
